## **Perinatal Death (fetal component) Models -- Modelo C**

**Purpose.** Third model of the thesis, built to test the modelling framework (same pipeline stages as Modelo A and Modelo B) on a new outcome. It is not a manuscript model.

**Outcome.** Fetal death identified from the mother's ICD codes (`outcome_fetal_death`): *strict* codes (stillbirth / intrauterine death: ICD-10 O36.4, Z37.1, Z37.4, Z37.7; ICD-9 656.4x, V27.1, V27.4, V27.7) **plus** *partial* codes (multiple pregnancy with one stillborn: Z37.3, Z37.6, O31.2; V27.3, V27.6). 154 admissions in the feasibility check (136 strict-only). Early neonatal deaths are **not** captured: MIMIC-IV has no validated mother-newborn linkage.

**Population.** The same 25,736 admissions of Modelo A and Modelo B (`mimiciv.cohort_smm_dataset`), so the three models share one cohort. Prevalence about 0.6%.

**Design decisions that differ from Modelo A/B**

1. **Leakage control.** The `gdg_*` diagnosis-group columns, the SMM columns and the delivery flag are not used as predictors: they come from the same coded diagnoses that define the outcome or from discharge-time codes. Only age, admission features, ICU stay and first ICU charts/labs remain.
2. **Patient-level split.** Hold-out split and every cross-validation use `StratifiedGroupKFold` grouped by `subject_id`, so no patient appears on both sides (45% of patients have more than one admission).
3. **Preprocessing fitted on the training part only** (winsorization bounds, imputation values, scalers), inside every fold of the cross-validation.
4. **Outcome sensitivity analyses** (strict-only, ICD-10 era, gestational age at least 22 weeks) are run on the hold-out set.

**Concept warning.** The fetal-death diagnosis is usually coded at the same admission in which the event is recognised. The model therefore estimates risk among admissions of pregnant women using admission-level information; it is not a prospective prenatal prediction.

**Extended feature set (version 2).** After exploring MIMIC-IV (`perinatal_feature_discovery_en.sql`), four blocks are added to the admission/ICU variables, all known at or before arrival or measured in the first 24 h from arrival:

- `history`: hypertension, diabetes, obesity and renal disease in **earlier** admissions of the same patient, and a fetal-death code at least 365 days earlier (a closer one is usually the same pregnancy).
- `cbc`: first hemoglobin, hematocrit, platelets, WBC, MCV and RDW (about 45% of admissions, 58% of events).
- `panel`: first creatinine, ALT, AST, uric acid, urea nitrogen and urine protein/creatinine (pre-eclampsia work-up; ordered selectively).
- `omr`: outpatient measurements from the 365 days **before** the admission date (OMR table): last and maximum blood pressure, last weight and BMI, number of outpatient dates with vitals and days since the last one. Coverage: blood pressure 61% of admissions, weight 57%, BMI 41%; coverage was not higher among events (no ordering signature), but slightly lower for weight and BMI. They are outpatient measurements in general, not necessarily prenatal visits.

Deliberately **not** used: microbiology (CMV/toxoplasma serology and tissue specimens follow a fetal death), coagulation (PT/INR/PTT, fibrinogen) and LDH (5-10 times more frequent among events: post-diagnosis work-up), diagnoses/procedures/DRG/treatments of the same admission, and counts of tests ordered. Marital status, first service, first care unit, ED arrival and number of prior admissions showed little or no signal. Section 9b measures what each block adds.

**Same-pregnancy rule.** Admissions within 180 days after an earlier admission that already carries a fetal-death code (the second admission of the same pregnancy, 5 events, and post-event readmissions, 12 non-events) are dropped (`POST_EVENT_WINDOW_DAYS`).

SQL: `sql/04_cohort_perinatal/fetal_death_dataset_en.sql` (builds `mimiciv.cohort_fetal_death_dataset`) and `sql/04_cohort_perinatal/fetal_death_features_en.sql` (builds `mimiciv.cohort_fetal_death_dataset_ext`, the table loaded here).

### **1. Libraries and configuration**

In [ ]:
import os
import sys
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import clone
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             recall_score, precision_score, f1_score, balanced_accuracy_score,
                             brier_score_loss)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

# Output folders (relative to notebooks/): tables -> data/, figures -> figures/
DATA_DIR = os.path.join("..", "data")
FIG_DIR = os.path.join("..", "figures")
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

RANDOM_STATE = 42
N_SPLITS = 5            # outer hold-out (1 fold = 20%) and cross-validation folds
N_JOBS = 2              # conservative: nested joblib workers caused MemoryError in Modelo A
FAST = False            # True = tiny hyperparameter grids, only to check the notebook runs end to end

# Feature blocks of the MAIN model (Section 9b compares them)
USE_BLOCKS = {"history": True, "cbc": True, "panel": True, "omr": True}
# Drop admissions that belong to the same pregnancy as an earlier fetal-death admission (None = keep all)
POST_EVENT_WINDOW_DAYS = 180

In [ ]:
# Add own libraries (root of the repository)
sys.path.append(os.path.abspath(".."))
import importlib
import libraries.database as db_lib
importlib.reload(db_lib)
engine = db_lib.get_connect_database()

### **2. Load the dataset**

`mimiciv.cohort_fetal_death_dataset_ext` is loaded with `db_lib.get_fetal_death_cohort_ext_query()`. The query does **not** select the `gdg_*` columns, the SMM columns or the delivery flag (leakage control, see the header).

In [ ]:
query = db_lib.get_fetal_death_cohort_ext_query()
df = pd.read_sql(query, engine)

# Same-pregnancy rule: drop admissions within POST_EVENT_WINDOW_DAYS after an earlier fetal-death admission
if POST_EVENT_WINDOW_DAYS is not None:
    in_window = df["days_since_prior_fd"] <= POST_EVENT_WINDOW_DAYS      # NaN -> False (kept)
    print(f"Dropped {int(in_window.sum())} admissions within {POST_EVENT_WINDOW_DAYS} days after an earlier "
          f"fetal-death admission ({int(df.loc[in_window, 'outcome_fetal_death'].sum())} of them events)")
    df = df.loc[~in_window].reset_index(drop=True)

assert df["hadm_id"].is_unique, "Expected one row per admission"
n_adm, n_pat = len(df), df["subject_id"].nunique()
n_ev, n_strict = int(df["outcome_fetal_death"].sum()), int(df["fd_strict"].sum())
print(f"Admissions: {n_adm:,} | patients: {n_pat:,}")
print(f"Events (strict + partial): {n_ev} ({df['outcome_fetal_death'].mean()*100:.3f}%) | strict only: {n_strict}")
print(f"Patients with an event: {df.loc[df['outcome_fetal_death']==1, 'subject_id'].nunique()}")
print()
print(pd.crosstab(df["admission_icd_version"], df["outcome_fetal_death"], margins=True))
print()
print("Gestational weeks (ICD-10 events with a Z3A code):")
print(df.loc[df["outcome_fetal_death"] == 1, "gest_weeks"].describe().round(1).to_string())

### **3. Sentinel values and impossible zeros**

Same rule as Modelo A/B: physiologically impossible zeros and the 999999 sentinel are set to missing before anything else. This step uses no statistic of the data, so it can be applied to the whole dataset without leakage.

In [ ]:
cols_zero_impossible = [
    "heart_rate", "respiratory_rate", "o2_saturation",
    "pressure_blood_mean", "pressure_blood_systolic", "pressure_blood_diastolic",
    "temperature_fahrenheit", "weight_admission", "glucose", "hemoglobine",
    "sodium", "potassium", "noionized_calcium", "magnesium", "chloride",
    "bicarbonate", "phosphorus", "creatine", "hematrocit", "lactic_acid",
    "height", "peep_set",
    "lab_hemoglobin", "lab_hematocrit", "lab_platelets", "lab_wbc", "lab_mcv", "lab_rdw",
    "lab_creatinine", "lab_alt", "lab_ast", "lab_uric_acid", "lab_urea_nitrogen", "lab_urine_prot_creat"
]
for col in cols_zero_impossible:
    if col in df.columns:
        df[col] = df[col].replace(0, np.nan)

for col in ["glucose", "hemoglobine", "noionized_calcium", "hematrocit"]:
    if col in df.columns:
        df[col] = df[col].replace(999999, np.nan)

df = df.reset_index(drop=True)

### **4. Feature assembly and leakage control**

Identifiers and outcome-related columns are set aside (they are **never** predictors): `outcome_fetal_death` (target), `fd_strict`, `admission_icd_version` and `gest_weeks` (used only for the sensitivity analyses in Section 10). Categorical variables are encoded as in Modelo B (grouped categories from `data/*_groups.xlsx`, then dummies).

In [ ]:
ID_COLS = ["subject_id", "hadm_id", "stay_id", "dob"]
META_COLS = ["outcome_fetal_death", "fd_strict", "admission_icd_version", "gest_weeks", "days_since_prior_fd"]

y_all = df["outcome_fetal_death"].astype(int)
groups_all = df["subject_id"]
meta = df[["subject_id", "hadm_id"] + META_COLS].copy()

X_raw = df.drop(columns=[c for c in ID_COLS + META_COLS if c in df.columns]).copy()
n0 = len(X_raw)

X_raw = X_raw.merge(pd.read_excel(os.path.join(DATA_DIR, "ethnicity_groups.xlsx")),
                    on="ethnicity", how="left").drop(columns="ethnicity")
X_raw = X_raw.merge(pd.read_excel(os.path.join(DATA_DIR, "language_groups.xlsx")),
                    on="language", how="left").drop(columns="language")
X_raw = X_raw.merge(pd.read_excel(os.path.join(DATA_DIR, "admission_location_groups.xlsx")),
                    on="admission_location", how="left").drop(columns="admission_location")
assert len(X_raw) == n0, "A group file has duplicated keys: the merge changed the number of rows"

X_raw["icu_stay"] = X_raw["icu_stay"].astype(int)

# Feature blocks (Section 9b compares them) and availability indicators (computed BEFORE imputation)
HIST_COLS = ["hx_hypertension", "hx_diabetes", "hx_obesity", "hx_renal_disease", "hx_prior_fetal_death_1y"]
CBC_COLS = ["lab_hemoglobin", "lab_hematocrit", "lab_platelets", "lab_wbc", "lab_mcv", "lab_rdw"]
PANEL_COLS = ["lab_creatinine", "lab_alt", "lab_ast", "lab_uric_acid", "lab_urea_nitrogen", "lab_urine_prot_creat"]
X_raw["cbc_available"] = X_raw[CBC_COLS].notna().any(axis=1).astype(int)
X_raw["panel_available"] = X_raw[PANEL_COLS].notna().any(axis=1).astype(int)
for c in HIST_COLS:
    X_raw[c] = X_raw[c].fillna(0).astype(int)
OMR_COLS = ["omr_sbp_last", "omr_dbp_last", "omr_sbp_max", "omr_dbp_max",
            "omr_weight_kg_last", "omr_bmi_last", "omr_n_visits_365", "omr_days_since_last"]
X_raw["omr_n_visits_365"] = X_raw["omr_n_visits_365"].fillna(0)
X_raw["omr_available"] = X_raw[["omr_sbp_last", "omr_weight_kg_last", "omr_bmi_last"]].notna().any(axis=1).astype(int)
BLOCK_COLS = {"history": HIST_COLS,
              "cbc": CBC_COLS + ["cbc_available"],
              "panel": PANEL_COLS + ["panel_available"],
              "omr": OMR_COLS + ["omr_available"]}
X_raw = pd.get_dummies(
    X_raw,
    columns=["ethnicity_group", "insurance", "language_group", "admission_type", "admission_location_group"],
    dtype=int,
)
X_raw.columns = [c.replace(" ", "_") for c in X_raw.columns]

constant_cols = [c for c in X_raw.columns if X_raw[c].nunique(dropna=False) <= 1]
if constant_cols:
    print(f"Dropping {len(constant_cols)} zero-variance columns: {constant_cols}")
    X_raw = X_raw.drop(columns=constant_cols)

# Leakage audit: nothing derived from the outcome or from the diagnosis list may be in X
forbidden_prefixes = ("gdg_", "outcome", "fd_", "is_delivery", "primary_smm", "gest_weeks", "days_since")
leaky = [c for c in X_raw.columns if c.startswith(forbidden_prefixes)]
assert not leaky, f"Leaky columns in X: {leaky}"

X_full = X_raw.copy()          # every block, used by the ablation in Section 9b


def select_cols(blocks):
    """Base columns (admission + ICU) plus the requested feature blocks."""
    off = {c for b, cols in BLOCK_COLS.items() if b not in blocks for c in cols}
    return [c for c in X_full.columns if c not in off]


MAIN_BLOCKS = [b for b, on in USE_BLOCKS.items() if on]
X_raw = X_full[select_cols(MAIN_BLOCKS)].copy()

print(f"Main model blocks: {MAIN_BLOCKS or 'none (base only)'}")
print(f"Predictors: {X_raw.shape[1]} | rows: {len(X_raw):,} | events: {int(y_all.sum())}")
print(f"Events per predictor (EPP): {y_all.sum() / X_raw.shape[1]:.2f}")
for b, cols in BLOCK_COLS.items():
    avail = X_full[cols[-1]].mean() if cols[-1].endswith("_available") else X_full[cols].any(axis=1).mean()
    print(f"  block {b:8s}: {len(cols)} columns, with data in {avail*100:.1f}% of admissions")

### **5. Patient-level hold-out split**

`StratifiedGroupKFold` keeps every patient entirely on one side and preserves the event rate. One of the five outer folds (about 20% of the patients) is the hold-out test set; the remaining 80% is used for tuning and fitting. Modelo A and Modelo B split by admission (declared as a limitation in the manuscript); here the leakage across admissions of the same patient is removed.

In [ ]:
outer = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
tr_idx, te_idx = next(outer.split(X_raw, y_all, groups_all))

assert set(groups_all.iloc[tr_idx]).isdisjoint(set(groups_all.iloc[te_idx])), "A patient is on both sides"

y_train_raw, y_test = y_all.iloc[tr_idx].reset_index(drop=True), y_all.iloc[te_idx].reset_index(drop=True)
groups_train = groups_all.iloc[tr_idx].reset_index(drop=True)
meta_test = meta.iloc[te_idx].reset_index(drop=True)

print(f"Train: {len(tr_idx):,} admissions ({y_train_raw.sum()} events, {y_train_raw.mean()*100:.3f}%)")
print(f"Test : {len(te_idx):,} admissions ({y_test.sum()} events, {y_test.mean()*100:.3f}%)")
print(f"Patients: train {groups_train.nunique():,} | test {groups_all.iloc[te_idx].nunique():,} (no overlap)")

### **6. Preprocessing fitted on the training part only**

Same rules as Modelo A/B, but every statistic is learned on the training rows and then applied to the test rows (and, in the cross-validation of Section 9, re-learned inside each fold):

- **Winsorization** at Q1 - 3 IQR / Q3 + 3 IQR (with a clinical lower bound), no row removed; ordinal scales are not winsorized.
- **Imputation:** median if |skew| > 1, otherwise mode.
- **Scaling:** `RobustScaler` if |skew| > 1, otherwise `MinMaxScaler`.

Most women have no ICU stay, about half have no hospital-wide labs and about 40% have no outpatient measurements, so many values are imputed. `icu_stay`, `cbc_available`, `panel_available` and `omr_available` tell the model which rows are imputed (the indicators are computed before imputation).

In [ ]:
LAB_COLS = [
    "heart_rate", "respiratory_rate", "o2_saturation",
    "pressure_blood_mean", "pressure_blood_systolic", "pressure_blood_diastolic",
    "temperature_fahrenheit", "weight_admission", "glucose", "hemoglobine",
    "sodium", "potassium", "noionized_calcium", "magnesium", "chloride",
    "bicarbonate", "phosphorus", "creatine", "hematrocit", "lactic_acid",
    "gcs_eye_opening", "gcs_verbal_response", "gcs_motor_response",
    "richmond_ras_scale", "peep_set", "height", "anion_gap"
]
ORDINAL = {"gcs_eye_opening", "gcs_verbal_response", "gcs_motor_response", "richmond_ras_scale", "peep_set"}
CLINICAL_MIN = {
    "heart_rate": 0, "respiratory_rate": 0, "o2_saturation": 0,
    "pressure_blood_mean": 0, "pressure_blood_systolic": 0, "pressure_blood_diastolic": 0,
    "temperature_fahrenheit": 90, "weight_admission": 0, "glucose": 0, "sodium": 100,
    "potassium": 0, "noionized_calcium": 0, "magnesium": 0, "chloride": 0,
    "bicarbonate": 0, "phosphorus": 0, "creatine": 0, "lactic_acid": 0,
}
EXTRA_NUM_COLS = CBC_COLS + PANEL_COLS + OMR_COLS      # numeric columns of the new blocks
WINSOR_COLS = [c for c in LAB_COLS + EXTRA_NUM_COLS if c not in ORDINAL]
CONT_COLS = LAB_COLS + EXTRA_NUM_COLS + ["age_at_admission", "icu_stay_count"]


def fit_prep(X_tr):
    """Learn winsorization bounds, imputation values and scalers on TRAINING rows only."""
    prep = {"bounds": {}, "fill": {}, "scaler": {}}
    Xc = X_tr.copy()
    for col in [c for c in WINSOR_COLS if c in Xc.columns]:
        q1, q3 = Xc[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        if not np.isfinite(iqr) or iqr == 0:
            continue
        lo = max(q1 - 3 * iqr, CLINICAL_MIN.get(col, q1 - 3 * iqr))
        hi = q3 + 3 * iqr
        prep["bounds"][col] = (lo, hi)
        Xc[col] = Xc[col].clip(lower=lo, upper=hi)
    for col in [c for c in CONT_COLS if c in Xc.columns]:
        s = Xc[col]
        if s.notna().sum() == 0:
            fill = 0.0
        else:
            sk = s.skew()
            if (not np.isfinite(sk)) or abs(sk) > 1:
                fill = s.median()
            else:
                m = s.mode()
                fill = m.iloc[0] if len(m) else s.median()
        prep["fill"][col] = fill
        Xc[col] = s.fillna(fill)
        sk = Xc[col].skew()
        scaler = RobustScaler() if (np.isfinite(sk) and abs(sk) > 1) else MinMaxScaler()
        scaler.fit(Xc[[col]])
        prep["scaler"][col] = scaler
    return prep


def transform_prep(X_in, prep):
    Xc = X_in.copy()
    for col, (lo, hi) in prep["bounds"].items():
        Xc[col] = Xc[col].clip(lower=lo, upper=hi)
    for col in prep["fill"]:
        Xc[col] = Xc[col].fillna(prep["fill"][col])
        Xc[col] = prep["scaler"][col].transform(Xc[[col]]).ravel()
    return Xc


prep_main = fit_prep(X_raw.iloc[tr_idx])
X_train = transform_prep(X_raw.iloc[tr_idx], prep_main).reset_index(drop=True)
X_test = transform_prep(X_raw.iloc[te_idx], prep_main).reset_index(drop=True)
y_train = y_train_raw

assert X_train.isna().sum().sum() == 0 and X_test.isna().sum().sum() == 0, "NaN left after preprocessing"
print(f"X_train {X_train.shape} | X_test {X_test.shape}")

### **7. Models**

Class imbalance (about 0.6%) is handled with class weights (`balanced_subsample`, `balanced`, `scale_pos_weight`), the same as Modelo A. Hyperparameters are tuned with `GridSearchCV` on **average precision** using patient-grouped inner folds. The tuned base learners (Random Forest, XGBoost, Extra Trees, Logistic Regression) feed a stacking ensemble with a logistic meta-learner, built on the same inner folds.

The inner tuning folds share the preprocessing learned on the whole training part; this only affects the choice of hyperparameters, never the hold-out test set.

In [ ]:
inner_splits = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
                    .split(X_train, y_train, groups_train))


def run_grid(model, param_grid):
    gs = GridSearchCV(model, param_grid, scoring="average_precision", cv=inner_splits,
                      n_jobs=N_JOBS, refit=True)
    gs.fit(X_train, y_train)
    print(f"  best CV AP = {gs.best_score_:.4f} | {gs.best_params_}")
    return gs.best_estimator_


def evaluate(y_true, proba, thr=0.5):
    y_true = np.asarray(y_true)
    pred = (np.asarray(proba) >= thr).astype(int)
    return {
        "AUC": roc_auc_score(y_true, proba),
        "AP": average_precision_score(y_true, proba),
        "Recall@0.5": recall_score(y_true, pred, zero_division=0),
        "Precision@0.5": precision_score(y_true, pred, zero_division=0),
        "F1@0.5": f1_score(y_true, pred, zero_division=0),
        "BalancedAcc@0.5": balanced_accuracy_score(y_true, pred),
        "Brier": brier_score_loss(y_true, proba),
    }


def pick(full, fast):
    return fast if FAST else full

In [ ]:
print("Random Forest")
best_rf = run_grid(
    RandomForestClassifier(class_weight="balanced_subsample", random_state=RANDOM_STATE),
    pick({"n_estimators": [300, 500], "max_depth": [5, 10, 20, None],
          "min_samples_split": [2, 5], "min_samples_leaf": [1, 2, 4], "max_features": ["sqrt"]},
         {"n_estimators": [100], "max_depth": [5], "min_samples_leaf": [2]}))

print("XGBoost")
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"  scale_pos_weight = {scale_pos_weight:.1f}")
best_xgb = run_grid(
    XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=RANDOM_STATE),
    pick({"n_estimators": [200, 400], "max_depth": [3, 5], "learning_rate": [0.01, 0.05, 0.1],
          "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]},
         {"n_estimators": [50], "max_depth": [3], "learning_rate": [0.1]}))

print("Extra Trees")
best_et = run_grid(
    ExtraTreesClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    pick({"n_estimators": [300, 500], "max_depth": [None, 20], "min_samples_split": [2, 5],
          "min_samples_leaf": [1, 2, 4], "max_features": ["sqrt"]},
         {"n_estimators": [100], "max_depth": [5], "min_samples_leaf": [2]}))

print("Logistic Regression")
best_lr = run_grid(
    LogisticRegression(class_weight="balanced", max_iter=3000),
    pick({"C": [0.01, 0.1, 1, 5, 10], "solver": ["lbfgs"]},
         {"C": [0.1, 1], "solver": ["lbfgs"]}))

In [ ]:
def make_stacking(rf, xgb, et, lr, splits):
    return StackingClassifier(
        estimators=[("rf", clone(rf)), ("xgb", clone(xgb)), ("et", clone(et)), ("rl", clone(lr))],
        final_estimator=LogisticRegression(C=0.1, solver="lbfgs", max_iter=5000,
                                           class_weight="balanced", random_state=RANDOM_STATE),
        cv=splits, n_jobs=N_JOBS)


stacking_clf = make_stacking(best_rf, best_xgb, best_et, best_lr, inner_splits)
stacking_clf.fit(X_train, y_train)

tuned = {
    "Random Forest": best_rf, "XGBoost": best_xgb, "Extra Trees": best_et,
    "Logistic Regression": best_lr, "Stacking": stacking_clf,
}
proba_test = {name: m.predict_proba(X_test)[:, 1] for name, m in tuned.items()}

### **8. Hold-out test results**

With only about 30 events in the test set, point estimates are unstable; 95% bootstrap intervals (resampling admissions) are reported for every model. The 0.5 threshold is shown for completeness; for a rare outcome the ranking metrics (AUC, AP) and the "flag the top x%" table are more informative.

In [ ]:
def boot_ci(y_true, proba, metric, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    vals = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y_true), len(y_true))
        if y_true[i].sum() in (0, len(i)):
            continue
        vals.append(metric(y_true[i], proba[i]))
    return np.percentile(vals, [2.5, 97.5])


rows = []
for name, p in proba_test.items():
    r = {"Model": name}
    r.update(evaluate(y_test, p))
    lo, hi = boot_ci(y_test, p, roc_auc_score)
    r["AUC 95% CI"] = f"[{lo:.3f}, {hi:.3f}]"
    lo, hi = boot_ci(y_test, p, average_precision_score)
    r["AP 95% CI"] = f"[{lo:.3f}, {hi:.3f}]"
    rows.append(r)
results_test = pd.DataFrame(rows).set_index("Model")
print(f"Test prevalence (AP baseline): {y_test.mean():.4f}")
print(results_test.round(3).to_string())
results_test.round(4).to_csv(os.path.join(DATA_DIR, "Table_Results_FetalDeath.csv"))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, p in proba_test.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_test, p):.3f})")
    pr, rc, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rc, pr, label=f"{name} (AP {average_precision_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--"); axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC -- fetal death")
axes[1].axhline(y_test.mean(), color="k", linestyle="--", label="Prevalence")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision"); axes[1].set_title("Precision-Recall -- fetal death")
axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Fig_ROC_PR_FetalDeath.png"), dpi=300, bbox_inches="tight")
plt.savefig(os.path.join(FIG_DIR, "Fig_ROC_PR_FetalDeath.pdf"), format="pdf", bbox_inches="tight")
plt.show()

cm = confusion_matrix(y_test, (proba_test["Stacking"] >= 0.5).astype(int))
ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues", colorbar=False)
plt.title("Stacking -- confusion matrix at 0.5")
plt.show()

In [ ]:
def topk_table(y_true, proba, ks=(0.01, 0.02, 0.05, 0.10, 0.20)):
    y_true = np.asarray(y_true)
    order = np.argsort(-np.asarray(proba))
    n, total_pos = len(y_true), int(y_true.sum())
    rows = []
    for k in ks:
        m = max(1, int(round(n * k)))
        tp = int(y_true[order[:m]].sum())
        rows.append({"flag top %": int(k * 100), "admissions flagged": m, "events caught": tp,
                     "recall": tp / total_pos if total_pos else np.nan,
                     "precision": tp / m, "lift": (tp / m) / (total_pos / n) if total_pos else np.nan})
    return pd.DataFrame(rows)


topk_df = topk_table(y_test, proba_test["Stacking"])
print("Stacking -- flag the top x% of admissions by predicted risk (hold-out test)")
print(topk_df.round(3).to_string(index=False))
topk_df.round(4).to_csv(os.path.join(DATA_DIR, "Table_TopK_FetalDeath.csv"), index=False)

# Calibration of the stacking probabilities (class weights inflate them: not absolute risks)
p = np.clip(proba_test["Stacking"], 1e-6, 1 - 1e-6)
logit = np.log(p / (1 - p)).reshape(-1, 1)
cal = LogisticRegression(C=1e6, max_iter=1000).fit(logit, y_test)
print(f"\nCalibration on the hold-out set: slope = {cal.coef_[0][0]:.3f}, intercept = {cal.intercept_[0]:.3f}, "
      f"Brier = {brier_score_loss(y_test, p):.4f} (reference Brier of a constant prevalence model: "
      f"{y_test.mean() * (1 - y_test.mean()):.4f})")
frac_pos, mean_pred = calibration_curve(y_test, p, n_bins=5, strategy="quantile")
plt.figure(figsize=(5, 4))
plt.plot(mean_pred, frac_pos, "o-", label="Stacking")
plt.plot([0, max(mean_pred.max(), frac_pos.max())], [0, max(mean_pred.max(), frac_pos.max())], "k--", label="Ideal")
plt.xlabel("Mean predicted probability"); plt.ylabel("Observed fraction"); plt.title("Calibration -- fetal death")
plt.legend(); plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Fig_Calibration_FetalDeath.png"), dpi=300, bbox_inches="tight")
plt.show()

### **9. Patient-grouped cross-validation (primary evidence)**

With so few events, a single hold-out is a weak basis. Here the whole dataset is evaluated with `StratifiedGroupKFold` (5 folds, grouped by patient). **Inside every fold** the preprocessing is re-learned on the training folds and the models are refitted with the hyperparameters selected in Section 7 (not re-tuned, so the numbers are slightly optimistic about tuning but not about leakage between patients or preprocessing). Pooled out-of-fold AUC and AP are reported with bootstrap intervals, plus the per-fold mean and standard deviation.

In [ ]:
def grouped_cv(make_model, X_in, y_in, groups_in, n_splits=N_SPLITS, seed=RANDOM_STATE):
    skf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.full(len(y_in), np.nan)
    fold_rows = []
    for k, (tr, te) in enumerate(skf.split(X_in, y_in, groups_in)):
        prep = fit_prep(X_in.iloc[tr])
        Xtr = transform_prep(X_in.iloc[tr], prep).reset_index(drop=True)
        Xte = transform_prep(X_in.iloc[te], prep).reset_index(drop=True)
        ytr = y_in.iloc[tr].reset_index(drop=True)
        gtr = groups_in.iloc[tr].reset_index(drop=True)
        model = make_model(Xtr, ytr, gtr)
        model.fit(Xtr, ytr)
        p = model.predict_proba(Xte)[:, 1]
        oof[te] = p
        yte = y_in.iloc[te]
        fold_rows.append({"fold": k + 1, "n_pos": int(yte.sum()),
                          "AUC": roc_auc_score(yte, p), "AP": average_precision_score(yte, p)})
    return pd.DataFrame(fold_rows), oof


def stacking_factory(Xtr, ytr, gtr):
    splits = list(StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
                  .split(Xtr, ytr, gtr))
    return make_stacking(best_rf, best_xgb, best_et, best_lr, splits)


factories = {
    "Random Forest": lambda Xtr, ytr, gtr: clone(best_rf),
    "XGBoost": lambda Xtr, ytr, gtr: clone(best_xgb),
    "Extra Trees": lambda Xtr, ytr, gtr: clone(best_et),
    "Logistic Regression": lambda Xtr, ytr, gtr: clone(best_lr),
    "Stacking": stacking_factory,
}

cv_rows, oof_store = [], {}
for name, factory in factories.items():
    folds, oof = grouped_cv(factory, X_raw, y_all, groups_all)
    oof_store[name] = oof
    lo_a, hi_a = boot_ci(y_all, oof, roc_auc_score, n_boot=500)
    lo_p, hi_p = boot_ci(y_all, oof, average_precision_score, n_boot=500)
    cv_rows.append({
        "Model": name,
        "AUC (mean +/- sd over folds)": f"{folds['AUC'].mean():.3f} +/- {folds['AUC'].std():.3f}",
        "AP (mean +/- sd over folds)": f"{folds['AP'].mean():.3f} +/- {folds['AP'].std():.3f}",
        "Pooled OOF AUC [95% CI]": f"{roc_auc_score(y_all, oof):.3f} [{lo_a:.3f}, {hi_a:.3f}]",
        "Pooled OOF AP [95% CI]": f"{average_precision_score(y_all, oof):.3f} [{lo_p:.3f}, {hi_p:.3f}]",
    })
    print(f"{name}: done")

cv_df = pd.DataFrame(cv_rows).set_index("Model")
print(f"\nPrevalence (AP baseline): {y_all.mean():.4f}")
print(cv_df.to_string())
cv_df.to_csv(os.path.join(DATA_DIR, "Table_CV_FetalDeath.csv"))

### **9b. Feature-block ablation (patient-grouped cross-validation)**

What does each block add? The same grouped cross-validation as Section 9 is run for XGBoost and Logistic Regression (hyperparameters from Section 7, tuned on the main feature set) with the base variables (admission attributes, ICU stay and charted values) and then adding blocks. If a block helps only through the *availability* indicators (a test being ordered), the pooled AUC/AP change but the values themselves add nothing: compare the "+ CBC" and "+ CBC + panel" rows with the base, and read the SHAP plot of Section 11. The OMR block is the one least exposed to that problem: its coverage was not higher among events and the records precede the admission.

In [ ]:
ablation_sets = {
    "Base (admission + ICU)": [],
    "+ history": ["history"],
    "+ CBC": ["cbc"],
    "+ CBC + panel": ["cbc", "panel"],
    "+ OMR (outpatient)": ["omr"],
    "All blocks": ["history", "cbc", "panel", "omr"],
}
abl_models = {"XGBoost": factories["XGBoost"], "Logistic Regression": factories["Logistic Regression"]}

abl_rows = []
for label, blocks in ablation_sets.items():
    cols = select_cols(blocks)
    for mname, factory in abl_models.items():
        folds, oof = grouped_cv(factory, X_full[cols], y_all, groups_all)
        lo_a, hi_a = boot_ci(y_all, oof, roc_auc_score, n_boot=500)
        lo_p, hi_p = boot_ci(y_all, oof, average_precision_score, n_boot=500)
        abl_rows.append({
            "Feature set": label, "Model": mname, "predictors": len(cols),
            "Pooled OOF AUC [95% CI]": f"{roc_auc_score(y_all, oof):.3f} [{lo_a:.3f}, {hi_a:.3f}]",
            "Pooled OOF AP [95% CI]": f"{average_precision_score(y_all, oof):.3f} [{lo_p:.3f}, {hi_p:.3f}]",
        })
    print(f"{label}: done")

ablation_df = pd.DataFrame(abl_rows)
print(f"\nPrevalence (AP baseline): {y_all.mean():.4f}")
print(ablation_df.to_string(index=False))
ablation_df.to_csv(os.path.join(DATA_DIR, "Table_Ablation_FetalDeath.csv"), index=False)

### **10. Outcome sensitivity analyses (hold-out test, stacking model)**

The model is trained on the main outcome (strict + partial codes). It is re-evaluated against alternative definitions, **without retraining**:

- **Strict only:** admissions whose only fetal-death code is partial (one liveborn twin) are removed.
- **ICD-10 era:** admissions coded in ICD-10 (from October 2015), where the Z3A codes exist.
- **ICD-10, at least 22 weeks:** ICD-10 admissions; events with a recorded gestation under 22 weeks are removed (events without a Z3A code are kept).
- **ICD-9 era:** admissions coded in ICD-9.

Results with fewer than 5 events are not reported.

In [ ]:
p_st = proba_test["Stacking"]
is_event = meta_test["outcome_fetal_death"].values == 1
partial_only = is_event & (meta_test["fd_strict"].values == 0)
icd10 = meta_test["admission_icd_version"].values == 10
icd9 = meta_test["admission_icd_version"].values == 9
under22 = is_event & (meta_test["gest_weeks"].values < 22)   # NaN compares as False: events without Z3A are kept

scenarios = {
    "Main outcome (strict + partial), all admissions": np.ones(len(y_test), dtype=bool),
    "Strict codes only (partial-only events removed)": ~partial_only,
    "ICD-10 admissions only": icd10,
    "ICD-10, events >= 22 weeks (events < 22 weeks removed)": icd10 & ~under22,
    "ICD-9 admissions only": icd9,
}
sens_rows = []
for label, mask in scenarios.items():
    yt, pp = y_test.values[mask], p_st[mask]
    n_pos = int(yt.sum())
    row = {"Scenario": label, "n": int(mask.sum()), "events": n_pos}
    if n_pos >= 5 and (len(yt) - n_pos) >= 5:
        row.update({"AUC": roc_auc_score(yt, pp), "AP": average_precision_score(yt, pp),
                    "AP baseline": yt.mean()})
    else:
        row.update({"AUC": np.nan, "AP": np.nan, "AP baseline": yt.mean() if len(yt) else np.nan})
    sens_rows.append(row)
sens_df = pd.DataFrame(sens_rows)
print(sens_df.round(3).to_string(index=False))
sens_df.round(4).to_csv(os.path.join(DATA_DIR, "Table_Sensitivity_FetalDeath.csv"), index=False)

### **11. Explainability**

SHAP values of the XGBoost base learner on the hold-out set (top 15 features), plus impurity-based importances of the Random Forest. These explain the base learners, not the stacking meta-learner.

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(best_xgb)
    shap_values = explainer.shap_values(X_test)
    shap.summary_plot(shap_values, X_test, max_display=15, show=False)
    plt.title("SHAP -- XGBoost base learner (fetal death)")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "Fig_SHAP_XGB_FetalDeath.png"), dpi=300, bbox_inches="tight")
    plt.show()
except Exception as exc:
    print(f"SHAP skipped ({type(exc).__name__}: {exc})")

imp = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
plt.figure(figsize=(6, 5))
imp[::-1].plot(kind="barh", color="steelblue")
plt.title("Random Forest -- top 15 importances (fetal death)")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "Fig_RF_Importance_FetalDeath.png"), dpi=300, bbox_inches="tight")
plt.show()

### **12. Limitations and open questions for the thesis**

- **Proxy outcome.** Fetal death is read from the mother's ICD codes; there is no chart review and no mother-newborn linkage, so early neonatal deaths (the rest of the perinatal period) are not observed. The outcome is best named *fetal death (perinatal component)*.
- **Mixed definitions.** The 22-week threshold can only be checked in ICD-10 admissions (Z3A codes). ICD-9 events cannot be verified and ICD-10 events below 22 weeks (probably 20-21 week losses) are kept in the main outcome; Section 10 shows the effect. *Partial* codes include multiple pregnancies in which one twin was liveborn.
- **Few events.** About 154 events and roughly 30 in the hold-out set: wide intervals; the grouped cross-validation is the primary evidence.
- **Diagnosis versus prediction.** The event is usually coded at the same admission, and the predictors are admission-level (age, admission type, insurance, ICU stay, first ICU charts and labs). No prenatal-care, ultrasound or obstetric-history variables exist in this cohort, so a weak signal is expected.
- **Imputed variables.** Most admissions have no ICU stay and about half have no hospital-wide labs; imputed values carry no information, and the availability indicators can reflect ordering patterns (selective ordering of the pre-eclampsia panel) instead of physiology. Section 9b separates what each block adds.
- **First 24 hours are not always before the diagnosis.** The window starts at arrival, but the fetal death may already be known; this is why microbiology, coagulation and LDH were excluded. Some residual post-diagnosis information in the hematology and chemistry values cannot be ruled out with these data.
- **Same-pregnancy rule.** 17 admissions within 180 days after an earlier fetal-death admission were dropped (5 events). Without this, 'prior fetal death' showed an odds ratio near 10 that came almost entirely from the same pregnancy; with a gap of at least 365 days there was 1 event in 78 admissions.
- **OMR is not necessarily prenatal.** It holds outpatient measurements of any kind in the 365 days before admission (pregnancy weight is mixed with pre-pregnancy weight, so BMI is the BMI at the last visit, not pre-pregnancy BMI). Fewer outpatient records among events may reflect less contact with the health system.
- **Not used.** Marital status, first service, first care unit, ED arrival and the number of prior admissions had little or no signal.
- **Probabilities are not absolute risks.** Class weights inflate them; a recalibration step (as in Modelo A/B) is needed before any risk-band use.
- **Single center and ICD-based;** no external validation. This notebook is research code, not a clinical tool.